# ==============================================================================
# 🚗 AUTOMOTIVE PRICING INTELLIGENCE: CAR PRICE PREDICTION VIA DECISION TREE REGRESSOR
# ==============================================================================
### Problem Formulation:
Vehicle pricing is determined by categorical configurations (fuel type, aspiration, body style, drive wheel) and physical engine metrics (horsepower, curb weight, engine size, highway mpg).
A Decision Tree Regressor recursively splits on technical thresholds:
$$\text{IF } \text{engine\_size} > 180 \text{ AND } \text{curb\_weight} > 3000 \implies \hat{y} = \$32,500$$


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Automotive Pricing Environment ready.")


✅ STEP 1: Automotive Pricing Environment ready.


## 📥 STEP 2 & 3: INGESTION & COLUMN CLEANUP
Loading Kaggle Car Price Assignment dataset (205 automotive models).


In [2]:
# STEP 2 & 3: LOAD & CLEANUP
data_path = 'data/car_price/CarPrice_Assignment.csv'
df = pd.read_csv(data_path)

# Drop ID and raw car name string
df = df.drop(columns=['car_ID', 'CarName'])
df.columns = [c.strip().lower() for c in df.columns]

print(f"📊 Dataset Shape: {df.shape[0]} vehicles, {df.shape[1]} attributes")
print(f"Columns: {list(df.columns)[:8]}")
df.head(3)


📊 Dataset Shape: 205 vehicles, 24 attributes
Columns: ['symboling', 'fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'wheelbase']


,symboling,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,3,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,3,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,1,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `price`.


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df.drop(columns=['price'])
y = df['price']

print(f"Missing attributes: {X.isnull().sum().sum()}")
print(f"Target Vehicle Price Summary ($):")
print(y.describe().round(2))


Missing attributes: 0
Target Vehicle Price Summary ($):
count      205.00
mean     13276.71
std       7988.85
min       5118.00
25%       7788.00
50%      10295.00
75%      16503.00
max      45400.00
Name: price, dtype: float64


## 🔒 STEP 6: LEAK-FREE TRAIN-TEST SPLIT
80/20 train/test partition.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Vehicles: {X_train.shape[0]}")
print(f"Test Vehicles : {X_test.shape[0]}")


Train Vehicles: 164
Test Vehicles : 41


## ⚙️ STEP 7: COLUMNTRANSFORMER FOR DECISION TREES
OneHotEncoder for categorical traits; numerical metrics pass through without scaling.


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', 'passthrough', num_cols)
])
print(f"Categorical features: {len(cat_cols)} | Numerical features: {len(num_cols)}")


Categorical features: 9 | Numerical features: 14


/tmp/ipykernel_632303/1949440638.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


## ⚠️ STEP 8: THE OVERFITTING TRAP
Evaluating unconstrained tree vs pruned tree.


In [6]:
# STEP 8: OVERFITTING DEMONSTRATION
pipe_uncon = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeRegressor(max_depth=None, random_state=42))
])
pipe_uncon.fit(X_train, y_train)

print(f"Unconstrained Tree Train R² : {r2_score(y_train, pipe_uncon.predict(X_train))*100:.2f}%")
print(f"Unconstrained Tree Test R²  : {r2_score(y_test, pipe_uncon.predict(X_test))*100:.2f}%")


Unconstrained Tree Train R² : 99.87%
Unconstrained Tree Test R²  : 88.81%


/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## 🎯 STEP 9: TUNING VIA 5-FOLD CROSS VALIDATION
Optimizing `max_depth` and `min_samples_leaf`.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'tree__max_depth': [3, 4, 5, 6, 8],
    'tree__min_samples_split': [2, 5, 10],
    'tree__min_samples_leaf': [2, 4, 8]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    Pipeline([('prep', preprocessor), ('tree', DecisionTreeRegressor(random_state=42))]),
    param_grid=param_grid,
    cv=cv,
    scoring='r2',
    n_jobs=-1
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV R²   : {grid.best_score_*100:.2f}%")


/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown 

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown cat

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown cat

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unkno

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown cat

/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown cat

🏆 Best Hyperparameters: {'tree__max_depth': 8, 'tree__min_samples_leaf': 4, 'tree__min_samples_split': 10}
🎯 Best 5-Fold CV R²   : 84.90%


## ⚔️ STEP 10: ALGORITHM DUEL (DecisionTreeRegressor vs Ridge)
Benchmarking DecisionTreeRegressor against regularized Ridge regression.


In [8]:
# STEP 10: MODEL BENCHMARK
from sklearn.preprocessing import StandardScaler

pipe_ridge = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', StandardScaler(), num_cols)
    ])),
    ('ridge', Ridge(alpha=10.0, random_state=42))
])
pipe_ridge.fit(X_train, y_train)

y_pred_tree = best_model.predict(X_test)
y_pred_ridge = pipe_ridge.predict(X_test)

print("="*65)
print(f"DecisionTreeRegressor (Tuned) -> Test R²: {r2_score(y_test, y_pred_tree)*100:.2f}% | RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_tree)):,.2f}")
print(f"Ridge Regression              -> Test R²: {r2_score(y_test, y_pred_ridge)*100:.2f}% | RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_ridge)):,.2f}")
print("="*65)


DecisionTreeRegressor (Tuned) -> Test R²: 91.35% | RMSE: $2,612.56
Ridge Regression              -> Test R²: 85.34% | RMSE: $3,402.40


/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## 📊 STEP 11: RESIDUAL ANALYSIS & PRICING METRICS
Diagnostic review of MSRP errors and vehicle pricing drivers.


In [9]:
# STEP 11: EVALUATION REPORT
mae = mean_absolute_error(y_test, y_pred_tree)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_tree))

print(f"📋 AUTOMOTIVE VALUATION METRICS:")
print(f"   Mean Absolute Error (MAE) : ${mae:,.2f}")
print(f"   Root Mean Squared Error   : ${rmse:,.2f}")
print(f"   R² Score                  : {r2_score(y_test, y_pred_tree)*100:.2f}%")


📋 AUTOMOTIVE VALUATION METRICS:
   Mean Absolute Error (MAE) : $1,837.10
   Root Mean Squared Error   : $2,612.56
   R² Score                  : 91.35%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & DEALERSHIP SMOKE TEST
Saving model pipeline and testing instant dealership inventory pricing latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/automotive_pricing_tree_pipeline.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_car = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_price = loaded_pipe.predict(sample_car)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🚗 LIVE DEALERSHIP INVENTORY PRICING:")
print(f"   Estimated Retail MSRP  : 🏷️ ${pred_price:,.2f}")
print(f"   Actual Listed MSRP     : ${y_test.iloc[0]:,.2f}")
print(f"   Pricing Latency        : {latency_ms:.3f} ms (SLA < 1.0ms: {'PASS' if latency_ms < 1.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/automotive_pricing_tree_pipeline.joblib (10,067 bytes)

🚗 LIVE DEALERSHIP INVENTORY PRICING:
   Estimated Retail MSRP  : 🏷️ $33,908.06
   Actual Listed MSRP     : $30,760.00
   Pricing Latency        : 16.283 ms (SLA < 1.0ms: CHECK)
